# Лабораторная работа №1

## Задание

***Ход лабораторной работы***

1. Составьте набор правил для предметной области, которая указана под Вашими ФИО.

2. Подумайте, какие факты вам потребуются для работы ваших правил.

3. Реализуйте алгоритмы прямого и обратного вывода.

4. Реализуйте разрешение конфликтов.

5. Проверьте работу алгоритмов на нескольких примерах.

6. Реализуйте примеры с помощью фреймворка

7. Проверьте работу алгоритмов на нескольких примерах.

***Мое задание***

Система планирования производственных операций (с учетом ограничений и приоритетов)

 Разработайте базу правил для системы, которая помогает планировать последовательность производственных операций на заводе. Система должна учитывать ограничения на оборудование, доступность сырья, сроки выполнения заказов и приоритеты различных заказов.

***Требования:***

•  База правил должна описывать минимум 4 типа производственных операций.

•  Каждая операция должна иметь набор требований к оборудованию и сырью.

•  Система должна учитывать приоритет заказов (например, срочные заказы должны выполняться раньше).

•  Реализуйте правила для разрешения конфликтов, когда несколько заказов требуют одного и того же ограниченного ресурса (оборудования или сырья) в одно и то же время.

•  Включите правила, учитывающие возможность параллельного выполнения операций, если это возможно.

## Выполнение

In [1]:
!pwd

/Users/artyem.petrov/dev/university/4-1/workshop-intelligent-systems/lab01


### 1. Подготовка к выполнению:

In [2]:
!which jupyter-lab

/opt/homebrew/Caskroom/miniconda/base/envs/legacy-ok-porn-detector/bin/jupyter-lab


In [3]:
!python --version

Python 3.13.11


In [4]:
!pip install experta

  Using cached experta-1.9.4-py3-none-any.whl.metadata (5.0 kB)
  Using cached frozendict-1.2-py3-none-any.whl
  Using cached schema-0.6.7-py2.py3-none-any.whl.metadata (14 kB)
Using cached experta-1.9.4-py3-none-any.whl (35 kB)
Using cached schema-0.6.7-py2.py3-none-any.whl (14 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [experta]


In [5]:
from typing import List, Dict, Any, Tuple, Set
from enum import Enum
from datetime import datetime, timedelta
import heapq

### 2. Определим типы операций: 

In [6]:
class OperationType(Enum):
    CUTTING = "cutting" # Резка
    WELDING = "welding" # Сварка 
    PAINTING = "painting" # Краска
    ASSEMBLY = "assembly" # Сборка

### 3. Определим класса для типа Заказ:

In [7]:
class Order:
    def __init__(self, order_id: int, priority: int, deadline: datetime):
        self.id = order_id
        self.priority = priority  # 1 - высший приоритет, 3 - низший
        self.deadline = deadline
        self.completion_time = None
        self.operations = []
    
    def __repr__(self) -> str:
        return f"Order({self.id}, priority={self.priority}, deadline={self.deadline})"

### 4. Определим класс для типа Операция: 

In [216]:
class ProductionOperation:
    def __init__(self, op_id: int, op_type: OperationType, order_id: int, 
                 duration: int, requirements: Dict[str, Any]):
        self.id = op_id
        self.type = op_type
        self.order_id = order_id
        self.duration = duration  # в часах
        self.requirements = requirements
        self.start_time = None
        self.end_time = None
        self.status = "pending"  # pending, scheduled, completed
    
    def __repr__(self) -> str:
        return f"Operation({self.id}, {self.type.value}, order={self.order_id}, duration={self.duration}h)"

### 5. Определим класс для Планировщика:

In [225]:
class ProductionPlanner:
    def __init__(self):
        # База фактов
        self.orders: Dict[int, Order] = {}
        self.operations: List[ProductionOperation] = []
        self.materials: Dict[str, int] = {}
        self.machines: Dict[str, List[Tuple[datetime, datetime]]] = {}
        self.scheduled_ops: List[ProductionOperation] = []
        
        # База правил
        self.rules = []
        self._initialize_rules()
        self._initialize_machines()
    
    def _initialize_machines(self):
        self.machines = {
            "cutting_machine": [],  # режущий станок
            "welding_machine": [],  # сварочный аппарат
            "painting_booth": [],   # покрасочная камера
            "assembly_table": []    # сборочный стол
        }
    
    def _initialize_rules(self):
        # Правило 1: Сначала выполняем операции с высоким приоритетом заказа
        self.rules.append({
            'name': 'high_priority_first',
            'condition': lambda op, facts: self.orders[op.order_id].priority == 1,
            'action': lambda op: setattr(op, 'priority_score', 100),
            'weight': 10
        })
        
        # Правило 2: Учитываем срок выполнения заказа
        self.rules.append({
            'name': 'deadline_urgency',
            'condition': lambda op, facts: True,
            'action': lambda op: setattr(op, 'deadline_score', 
                (self.orders[op.order_id].deadline - datetime.now()).days),
            'weight': 8
        })
        
        # Правило 3: Предпочтение коротким операциям для лучшего использования ресурсов
        self.rules.append({
            'name': 'short_operations_first',
            'condition': lambda op, facts: True,
            'action': lambda op: setattr(op, 'duration_score', 50 - op.duration),
            'weight': 5
        })
        
        # Правило 4: Параллельное выполнение разных типов операций
        self.rules.append({
            'name': 'parallel_possible',
            'condition': lambda op, facts: self._check_parallel_possible(op, facts),
            'action': lambda op: setattr(op, 'parallel_score', 20),
            'weight': 7
        })
        
        # Правило 5: Учитываем доступность сырья
        self.rules.append({
            'name': 'material_available',
            'condition': lambda op, facts: self._check_materials_available(op),
            'action': lambda op: setattr(op, 'material_score', 30),
            'weight': 9
        })
        
        # Правило 6: Минимизация перенастройки оборудования
        self.rules.append({
            'name': 'minimize_setup',
            'condition': lambda op, facts: self._check_similar_operations_scheduled(op),
            'action': lambda op: setattr(op, 'setup_score', 15),
            'weight': 6
        })
    
    def add_order(self, order: Order):
        self.orders[order.id] = order
    
    def add_operation(self, operation: ProductionOperation):
        self.operations.append(operation)
        self.orders[operation.order_id].operations.append(operation)
    
    def set_materials(self, materials: Dict[str, int]):
        self.materials = materials
    
    def _check_materials_available(self, operation: ProductionOperation) -> bool:
        if 'materials' not in operation.requirements:
            return True
        
        required_materials = operation.requirements['materials']
        for material, quantity in required_materials.items():
            if self.materials.get(material, 0) < quantity:
                return False
        return True
    
    def _check_parallel_possible(self, operation: ProductionOperation, facts: Dict) -> bool:
        # Операции разных типов можно выполнять параллельно
        scheduled_types = {op.type for op in facts.get('scheduled', [])}
        return operation.type not in scheduled_types
    
    def _check_similar_operations_scheduled(self, operation: ProductionOperation) -> bool:
        for op in self.scheduled_ops:
            if op.type == operation.type:
                return True
        return False
    
    def _is_machine_available(self, machine_name: str, start_time: datetime, 
                             duration: int) -> bool:
        if machine_name not in self.machines:
            return False
        
        end_time = start_time + timedelta(hours=duration)
        for busy_start, busy_end in self.machines[machine_name]:
            if not (end_time <= busy_start or start_time >= busy_end):
                return False
        return True
    
    def _reserve_machine(self, machine_name: str, start_time: datetime, 
                        duration: int):
        end_time = start_time + timedelta(hours=duration)
        self.machines[machine_name].append((start_time, end_time))
        # Сортируем по времени начала
        self.machines[machine_name].sort(key=lambda x: x[0])
    
    def _consume_materials(self, operation: ProductionOperation):
        if 'materials' in operation.requirements:
            for material, quantity in operation.requirements['materials'].items():
                self.materials[material] -= quantity

    # Прямой вывод: планирование операций на основе правил
    def forward_chaining(self, start_time: datetime = None):
        if start_time is None:
            start_time = datetime.now()
        
        current_time = start_time
        unscheduled_ops = self.operations.copy()
        
        while unscheduled_ops:
            # Сбор фактов
            facts = {
                'current_time': current_time,
                'scheduled': self.scheduled_ops.copy(),
                'available_materials': self.materials.copy(),
                'machine_schedule': self.machines.copy()
            }
            
            # Применение правил для выбора следующей операции
            candidate_ops = []
            
            for operation in unscheduled_ops:
                if not self._check_materials_available(operation):
                    continue
                
                # Применяем правила для вычисления приоритета
                total_score = 0
                
                for rule in self.rules:
                    if rule['condition'](operation, facts):
                        rule['action'](operation)
                        # Вес правила влияет на значимость оценки
                        score_attr = rule['name'].replace('_', '') + '_score'
                        if hasattr(operation, score_attr):
                            total_score += getattr(operation, score_attr) * rule['weight']
                
                # Определяем требуемое оборудование
                machine_map = {
                    OperationType.CUTTING: 'cutting_machine',
                    OperationType.WELDING: 'welding_machine',
                    OperationType.PAINTING: 'painting_booth',
                    OperationType.ASSEMBLY: 'assembly_table'
                }
                
                required_machine = machine_map[operation.type]
                
                candidate_ops.append({
                    'operation': operation,
                    'score': total_score,
                    'machine': required_machine,
                    'deadline': self.orders[operation.order_id].deadline
                })
            
            if not candidate_ops:
                break
            
            # Разрешение конфликтов (выбор лучшей операции)
            # Используем сложную эвристику для выбора
            best_op_info = self._resolve_conflicts(candidate_ops, facts)
            best_op = best_op_info['operation']
            
            # Планирование выбранной операции
            machine_name = best_op_info['machine']
            
            # Ищем ближайшее доступное время для оборудования
            scheduled_time = self._find_earliest_available_time(
                machine_name, current_time, best_op.duration)
            
            # Планируем операцию
            best_op.start_time = scheduled_time
            best_op.end_time = scheduled_time + timedelta(hours=best_op.duration)
            best_op.status = "scheduled"
            
            # Резервируем ресурсы
            self._reserve_machine(machine_name, scheduled_time, best_op.duration)
            self._consume_materials(best_op)
            
            self.scheduled_ops.append(best_op)
            unscheduled_ops.remove(best_op)
            
            print(f"\nЗапланирована операция: {best_op}")
            print(f"  Время: {best_op.start_time.strftime('%Y-%m-%d %H:%M')} - "
                  f"{best_op.end_time.strftime('%Y-%m-%d %H:%M')}")
            print(f"  Оборудование: {machine_name}")
            print(f"  Оценка приоритета: {best_op_info['score']:.1f}")
            
            # Обновляем время для следующей итерации
            if scheduled_time > current_time:
                current_time = scheduled_time
        
        return self.scheduled_ops

    # Обратный вывод: поиск плана для конкретного заказа
    def backward_chaining(self, goal_order_id: int, current_time: datetime = None):
        if current_time is None:
            current_time = datetime.now()
        
        goal_order = self.orders.get(goal_order_id)
        if not goal_order:
            print(f"Заказ {goal_order_id} не найден")
            return []
        
        # Находим все операции для этого заказа
        order_operations = [op for op in self.operations 
                          if op.order_id == goal_order_id]
        
        if not order_operations:
            print(f"Для заказа {goal_order_id} нет операций")
            return []
        
        # Строим план в обратном порядке
        plan = []
        required_materials = set()
        required_machines = set()
        
        machine_map = {
            OperationType.CUTTING: 'cutting_machine',
            OperationType.WELDING: 'welding_machine',
            OperationType.PAINTING: 'painting_booth',
            OperationType.ASSEMBLY: 'assembly_table'
        }
        
        for operation in order_operations:
            # Проверяем требования
            if 'materials' in operation.requirements:
                for material in operation.requirements['materials']:
                    required_materials.add(material)
            
            machine = machine_map[operation.type]
            required_machines.add(machine)
            
            plan.append({
                'operation': operation,
                'required_machine': machine,
                'requirements': operation.requirements
            })
        
        # Выводим информацию
        print(f"\nЗаказ: {goal_order}")
        print(f"Дедлайн: {goal_order.deadline}")
        print(f"Приоритет: {goal_order.priority}")
        
        print("\nТребуемые операции:")
        for i, item in enumerate(plan, 1):
            op = item['operation']
            print(f"  {i}. {op.type.value} - {op.duration}ч "
                  f"(требует: {item['required_machine']})")
        
        print("\nТребуемые материалы:")
        for material in required_materials:
            available = self.materials.get(material, 0)
            print(f"  - {material}: требуется {available}+ единиц")
        
        print("\nТребуемое оборудование:")
        for machine in required_machines:
            print(f"  - {machine}")
        
        # Проверяем возможность выполнения
        can_execute = True
        issues = []
        
        for material in required_materials:
            if self.materials.get(material, 0) <= 0:
                can_execute = False
                issues.append(f"Недостаточно материала: {material}")
        
        if not can_execute:
            print(f"\nВНИМАНИЕ: Заказ {goal_order_id} не может быть выполнен!")
            for issue in issues:
                print(f"  - {issue}")
        else:
            print(f"\nЗаказ {goal_order_id} может быть выполнен при наличии оборудования")
        
        
        return plan
    
    def _resolve_conflicts(self, candidate_ops: List[Dict], facts: Dict) -> Dict:
        """
        Разрешение конфликтов при выборе следующей операции
        Используется сложная эвристика с учетом:
        1. Приоритета заказа
        2. Срочности дедлайна
        3. Длительности операции
        4. Доступности ресурсов
        """
        # Улучшенная функция оценки
        for op_info in candidate_ops:
            op = op_info['operation']
            order = self.orders[op.order_id]
            
            # Базовый счет на основе приоритета
            priority_score = (4 - order.priority) * 25  # 1->75, 2->50, 3->25
            
            # Счет на основе дедлайна (чем ближе дедлайн, тем выше приоритет)
            days_to_deadline = (order.deadline - facts['current_time']).days
            if days_to_deadline <= 0:
                deadline_score = 100
            elif days_to_deadline <= 2:
                deadline_score = 80
            elif days_to_deadline <= 5:
                deadline_score = 50
            else:
                deadline_score = 20
            
            # Предпочтение коротким операциям
            duration_score = max(0, 40 - op.duration * 2)
            
            # Штраф за ожидание оборудования
            wait_penalty = 0
            machine_name = op_info['machine']
            if not self._is_machine_available(machine_name, facts['current_time'], op.duration):
                # Оцениваем время ожидания
                wait_penalty = -10
            
            # Итоговый счет
            final_score = (
                priority_score * 0.3 +
                deadline_score * 0.4 +
                duration_score * 0.2 +
                wait_penalty * 0.1
            )
            
            op_info['score'] = final_score
        
        # Выбираем операцию с максимальным счетом
        best_op = max(candidate_ops, key=lambda x: x['score'])
        
        # Логирование выбора
        print(f"\n[Разрешение конфликтов] Выбрана операция {best_op['operation'].id} "
              f"со счетом {best_op['score']:.1f}")
        
        return best_op

    # Находим ближайшее доступное время для оборудования
    def _find_earliest_available_time(self, machine_name: str, 
                                     start_from: datetime, 
                                     duration: int) -> datetime:
        if machine_name not in self.machines:
            return start_from
        
        busy_slots = self.machines[machine_name]
        
        if not busy_slots:
            return start_from
        
        # Проверяем возможность начать сразу
        end_time = start_from + timedelta(hours=duration)
        if self._is_machine_available(machine_name, start_from, duration):
            return start_from
        
        # Ищем ближайший доступный слот
        current_time = start_from
        
        # Проверяем промежутки между занятыми слотами
        for i in range(len(busy_slots)):
            busy_end = busy_slots[i][1]
            
            # Время после окончания текущего слота
            slot_start = busy_end
            
            # Проверяем, есть ли следующий слот
            if i + 1 < len(busy_slots):
                next_busy_start = busy_slots[i + 1][0]
                slot_end = next_busy_start
            else:
                slot_end = slot_start + timedelta(hours=24)  # До конца дня
            
            # Если слот достаточно длинный
            if (slot_end - slot_start) >= timedelta(hours=duration):
                return slot_start
        
        # Если не нашли, возвращаем время после последнего слота
        return busy_slots[-1][1]
    
    def print_schedule(self):        
        if not self.scheduled_ops:
            print("Нет запланированных операций")
            return
        
        # Сортируем по времени начала
        sorted_ops = sorted(self.scheduled_ops, key=lambda x: x.start_time)
        
        for op in sorted_ops:
            order = self.orders[op.order_id]
            machine_map = {
                OperationType.CUTTING: 'cutting_machine',
                OperationType.WELDING: 'welding_machine',
                OperationType.PAINTING: 'painting_booth',
                OperationType.ASSEMBLY: 'assembly_table'
            }
            
            print(f"\n{op.type.value.upper()} (Операция {op.id})")
            print(f"  Заказ: {order.id}, Приоритет: {order.priority}")
            print(f"  Время: {op.start_time.strftime('%Y-%m-%d %H:%M')} - "
                  f"{op.end_time.strftime('%Y-%m-%d %H:%M')}")
            print(f"  Длительность: {op.duration}ч")
            print(f"  Оборудование: {machine_map[op.type]}")
            
            if 'materials' in op.requirements:
                print(f"  Материалы: {op.requirements['materials']}")

### 6. Протестируем поведение: 

In [226]:
# Создаем планировщик
planner = ProductionPlanner()

# Устанавливаем начальные материалы
planner.set_materials({
    'metal': 100,
    'electrodes': 50,
    'paint': 30,
    'components': 200
})

# Создаем заказы
now = datetime.now()

orders = [
    Order(order_id=1, priority=1, deadline=now + timedelta(days=1)),  # Срочный заказ
    Order(2, 2, now + timedelta(days=3)),
    Order(3, 3, now + timedelta(days=5)),
    Order(4, 1, now + timedelta(days=2)),  # Еще один срочный
]

for order in orders:
    planner.add_order(order)

# Создаем производственные операции
operations = [
    # Заказ 1 (срочный)
    ProductionOperation(op_id=1, op_type=OperationType.CUTTING, order_id=1, duration=2, 
                       requirements={'materials': {'metal': 10}}),
    ProductionOperation(2, OperationType.WELDING, 1, 3,
                       {'materials': {'electrodes': 5}}),
    ProductionOperation(3, OperationType.ASSEMBLY, 1, 4,
                       {'materials': {'components': 20}}),
    
    # Заказ 2
    ProductionOperation(4, OperationType.CUTTING, 2, 3,
                       {'materials': {'metal': 15}}),
    ProductionOperation(5, OperationType.PAINTING, 2, 2,
                       {'materials': {'paint': 8}}),
    
    # Заказ 3
    ProductionOperation(6, OperationType.WELDING, 3, 5,
                       {'materials': {'electrodes': 10}}),
    ProductionOperation(7, OperationType.ASSEMBLY, 3, 3,
                       {'materials': {'components': 30}}),
    
    # Заказ 4 (срочный)
    ProductionOperation(8, OperationType.PAINTING, 4, 2,
                       {'materials': {'paint': 5}}),
    ProductionOperation(9, OperationType.ASSEMBLY, 4, 2,
                       {'materials': {'components': 15}}),
]

for op in operations:
    planner.add_operation(op)

print("\nИнициализировано:")
print(f"  - Заказов: {len(planner.orders)}")
print(f"  - Операций: {len(planner.operations)}")
print(f"  - Правил: {len(planner.rules)}")


Инициализировано:
  - Заказов: 4
  - Операций: 9
  - Правил: 6


### 7. Тестируем обратный вывод для конкретного заказа: 

In [227]:
planner.backward_chaining(1)  # Анализ заказа 1


Заказ: Order(1, priority=1, deadline=2025-12-16 20:34:39.760627)
Дедлайн: 2025-12-16 20:34:39.760627
Приоритет: 1

Требуемые операции:
  1. cutting - 2ч (требует: cutting_machine)
  2. welding - 3ч (требует: welding_machine)
  3. assembly - 4ч (требует: assembly_table)

Требуемые материалы:
  - components: требуется 200+ единиц
  - metal: требуется 100+ единиц
  - electrodes: требуется 50+ единиц

Требуемое оборудование:
  - welding_machine
  - assembly_table
  - cutting_machine

Заказ 1 может быть выполнен при наличии оборудования


[{'operation': Operation(1, cutting, order=1, duration=2h),
  'required_machine': 'cutting_machine',
  'requirements': {'materials': {'metal': 10}}},
 {'operation': Operation(2, welding, order=1, duration=3h),
  'required_machine': 'welding_machine',
  'requirements': {'materials': {'electrodes': 5}}},
 {'operation': Operation(3, assembly, order=1, duration=4h),
  'required_machine': 'assembly_table',
  'requirements': {'materials': {'components': 20}}}]

### 8. Тестируем прямой вывод:

In [228]:
scheduled_ops = planner.forward_chaining(now)


[Разрешение конфликтов] Выбрана операция 1 со счетом 61.7

Запланирована операция: Operation(1, cutting, order=1, duration=2h)
  Время: 2025-12-15 20:34 - 2025-12-15 22:34
  Оборудование: cutting_machine
  Оценка приоритета: 61.7

[Разрешение конфликтов] Выбрана операция 8 со счетом 61.7

Запланирована операция: Operation(8, painting, order=4, duration=2h)
  Время: 2025-12-15 20:34 - 2025-12-15 22:34
  Оборудование: painting_booth
  Оценка приоритета: 61.7

[Разрешение конфликтов] Выбрана операция 9 со счетом 61.7

Запланирована операция: Operation(9, assembly, order=4, duration=2h)
  Время: 2025-12-15 20:34 - 2025-12-15 22:34
  Оборудование: assembly_table
  Оценка приоритета: 61.7

[Разрешение конфликтов] Выбрана операция 2 со счетом 61.3

Запланирована операция: Operation(2, welding, order=1, duration=3h)
  Время: 2025-12-15 20:34 - 2025-12-15 23:34
  Оборудование: welding_machine
  Оценка приоритета: 61.3

[Разрешение конфликтов] Выбрана операция 3 со счетом 59.9

Запланирована оп

### 9. Выводим итоговое расписание

In [229]:
planner.print_schedule()


CUTTING (Операция 1)
  Заказ: 1, Приоритет: 1
  Время: 2025-12-15 20:34 - 2025-12-15 22:34
  Длительность: 2ч
  Оборудование: cutting_machine
  Материалы: {'metal': 10}

PAINTING (Операция 8)
  Заказ: 4, Приоритет: 1
  Время: 2025-12-15 20:34 - 2025-12-15 22:34
  Длительность: 2ч
  Оборудование: painting_booth
  Материалы: {'paint': 5}

ASSEMBLY (Операция 9)
  Заказ: 4, Приоритет: 1
  Время: 2025-12-15 20:34 - 2025-12-15 22:34
  Длительность: 2ч
  Оборудование: assembly_table
  Материалы: {'components': 15}

WELDING (Операция 2)
  Заказ: 1, Приоритет: 1
  Время: 2025-12-15 20:34 - 2025-12-15 23:34
  Длительность: 3ч
  Оборудование: welding_machine
  Материалы: {'electrodes': 5}

ASSEMBLY (Операция 3)
  Заказ: 1, Приоритет: 1
  Время: 2025-12-15 22:34 - 2025-12-16 02:34
  Длительность: 4ч
  Оборудование: assembly_table
  Материалы: {'components': 20}

PAINTING (Операция 5)
  Заказ: 2, Приоритет: 2
  Время: 2025-12-15 22:34 - 2025-12-16 00:34
  Длительность: 2ч
  Оборудование: painting_

### 10. Анализ использования ресурсов:

In [230]:
print(f"\nОстатки материалов:")
for material, quantity in planner.materials.items():
    print(f"  {material}: {quantity} единиц")

print(f"\nЗанятость оборудования:")
for machine, slots in planner.machines.items():
    total_hours = sum((end - start).total_seconds() / 3600 
                     for start, end in slots)
    print(f"  {machine}: {len(slots)} слотов, всего {total_hours:.1f} ч")


Остатки материалов:
  metal: 75 единиц
  electrodes: 35 единиц
  paint: 17 единиц
  components: 135 единиц

Занятость оборудования:
  cutting_machine: 2 слотов, всего 5.0 ч
  welding_machine: 2 слотов, всего 8.0 ч
  painting_booth: 2 слотов, всего 4.0 ч
  assembly_table: 3 слотов, всего 9.0 ч


### 11. Тестирование конфликтных ситуаций: 

Создаем конфликтную ситуацию: два заказа требуют один ресурс

In [231]:
# Добавляем еще операцию резки для другого срочного заказа
conflict_op = ProductionOperation(10, OperationType.CUTTING, 4, 4,
                                 {'materials': {'metal': 20}})
planner.add_operation(conflict_op)

# Запускаем планирование снова
planner.forward_chaining(now)


[Разрешение конфликтов] Выбрана операция 1 со счетом 60.7

Запланирована операция: Operation(1, cutting, order=1, duration=2h)
  Время: 2025-12-16 01:34 - 2025-12-16 03:34
  Оборудование: cutting_machine
  Оценка приоритета: 60.7

[Разрешение конфликтов] Выбрана операция 2 со счетом 68.3

Запланирована операция: Operation(2, welding, order=1, duration=3h)
  Время: 2025-12-15 23:34 - 2025-12-16 02:34
  Оборудование: welding_machine
  Оценка приоритета: 68.3

[Разрешение конфликтов] Выбрана операция 3 со счетом 67.9

Запланирована операция: Operation(3, assembly, order=1, duration=4h)
  Время: 2025-12-16 05:34 - 2025-12-16 09:34
  Оборудование: assembly_table
  Оценка приоритета: 67.9

[Разрешение конфликтов] Выбрана операция 8 со счетом 61.7

Запланирована операция: Operation(8, painting, order=4, duration=2h)
  Время: 2025-12-16 05:34 - 2025-12-16 07:34
  Оборудование: painting_booth
  Оценка приоритета: 61.7

[Разрешение конфликтов] Выбрана операция 10 со счетом 60.9

Запланирована о

[Operation(1, cutting, order=1, duration=2h),
 Operation(8, painting, order=4, duration=2h),
 Operation(9, assembly, order=4, duration=2h),
 Operation(2, welding, order=1, duration=3h),
 Operation(3, assembly, order=1, duration=4h),
 Operation(5, painting, order=2, duration=2h),
 Operation(4, cutting, order=2, duration=3h),
 Operation(7, assembly, order=3, duration=3h),
 Operation(6, welding, order=3, duration=5h),
 Operation(1, cutting, order=1, duration=2h),
 Operation(2, welding, order=1, duration=3h),
 Operation(3, assembly, order=1, duration=4h),
 Operation(8, painting, order=4, duration=2h),
 Operation(10, cutting, order=4, duration=4h),
 Operation(9, assembly, order=4, duration=2h),
 Operation(5, painting, order=2, duration=2h),
 Operation(4, cutting, order=2, duration=3h),
 Operation(6, welding, order=3, duration=5h),
 Operation(7, assembly, order=3, duration=3h)]

### 12. Проверка принципов планирования: 

In [232]:
# Проверяем, что срочные заказа выполнены первыми
high_priority_ops = [op for op in planner.scheduled_ops 
                    if planner.orders[op.order_id].priority == 1]

print(f"\nСрочные заказы запланированы первыми: "
      f"{len(high_priority_ops)} операций высокого приоритета")


# Проверяем параллельное выполнение
scheduled_by_type = {}
for op in planner.scheduled_ops:
    if op.type not in scheduled_by_type:
        scheduled_by_type[op.type] = []
    scheduled_by_type[op.type].append(op)

print(f"\nТипы операций в расписании:")
for op_type, ops in scheduled_by_type.items():
    print(f"  {op_type.value}: {len(ops)} операций")


Срочные заказы запланированы первыми: 11 операций высокого приоритета

Типы операций в расписании:
  cutting: 5 операций
  painting: 4 операций
  assembly: 6 операций
  welding: 4 операций


## Вывод: 

Мы научились проектировать и реализовывать экспертные системы для производственного планирования, включая базы правил, алгоритмы прямого и обратного вывода с учетом ограничений оборудования, доступности сырья и приоритетов заказов.